# 07 — Crop extraction and gallery construction

**Project:** RAOD++ · **Phase:** F6 · **Protocol:** v1.2

Turns the calibrated uncertainty band into the two crop sets the retrieval stage needs:

- **queries** — low-confidence regions mined from the training split, one set per seed
  (protocol v1.2 A6, Option A: seed-matched)
- **gallery** — crops from VisDrone2019-DET **test-dev**, images only. Its labels are never
  read and it is never evaluated on (protocol v1.2 A7).

CLIP embedding and the FAISS index come in notebook 08; this notebook only writes pixels
and manifests.

**Runtime:** band calibration ~15 min · query crops ~50 min · gallery ~20 min.
**Disk:** ~20 GB for queries, ~1 GB for the gallery.


## [Cell 3] Config

In [ ]:
# ===== Cell 3 =====
from pathlib import Path
import numpy as np, pandas as pd, yaml

CONFIG = {
    "PROJECT_ROOT": Path.home() / "RAODpp",
    "EXP": "E0",
    "DATASET": "visdrone",
    "SEEDS": [42, 123, 2024, 7, 3407, 11, 55, 101, 777, 12345],
    "IMGSZ": 640,
    "DEVICE": 0,
    "ALPHA": 0.20,            # context margin (protocol v1.2 A6)
    "CROP_SIZE": 224,         # CLIP ViT-B/32 input
    "JPEG_Q": 90,
    "PREC_LOW": 0.50,         # band lower edge: precision reaches 50%
    "PREC_HIGH": 0.90,        # band upper edge: precision reaches 90%
    "IOU_MATCH": 0.5,
    "CAL_IMAGES": 1500,       # images used for band calibration
    "MIN_BIN": 30,            # ignore precision bins with too little evidence
    "MIN_CROP_PX": 8,         # discard degenerate boxes
    "GALLERY_URL": "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip",
    "GALLERY_CONF": 0.25,     # gallery crops come from plausible detections
    "GALLERY_MAX_DET": 300,
}

ROOT = CONFIG["PROJECT_ROOT"]
OUT = ROOT/"output"; FIGDATA = OUT/"figdata"
RUNS = ROOT/"model"/"runs"
CROPS = ROOT/"dataset"/"crops"; CROPS.mkdir(parents=True, exist_ok=True)
DATA_YAML = ROOT/"dataset"/"visdrone_raodpp.yaml"

dcfg = yaml.safe_load(open(DATA_YAML))
BASE = Path(dcfg["path"]); NAMES = dcfg["names"]
TRAIN_IMG = BASE/dcfg["train"]
TRAIN_LBL = Path(str(TRAIN_IMG).replace("/images","/labels"))

GRID = np.array([0.05,0.075,0.10,0.15,0.20,0.25,0.30,0.40,0.50,0.60,0.70,0.80,0.90])

print("crops dir :", CROPS)
print("seeds     :", len(CONFIG["SEEDS"]))
print("band edges: precision", CONFIG["PREC_LOW"], "->", CONFIG["PREC_HIGH"])


## [Cell 5] Band calibration for every seed
Protocol v1.2 A5 requires the band to be calibrated per seed, not borrowed from one model.
Uses the cached predictions from notebook 06, so no inference is repeated.

In [ ]:
# ===== Cell 5 =====
from PIL import Image

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:,None,0], b[None,:,0]); y1 = np.maximum(a[:,None,1], b[None,:,1])
    x2 = np.minimum(a[:,None,2], b[None,:,2]); y2 = np.minimum(a[:,None,3], b[None,:,3])
    inter = np.clip(x2-x1, 0, None) * np.clip(y2-y1, 0, None)
    ar_a = (a[:,2]-a[:,0])*(a[:,3]-a[:,1]); ar_b = (b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    return inter / (ar_a[:,None] + ar_b[None,:] - inter + 1e-9)

# ground truth cached once
gt_cache = {}
for p in sorted(TRAIN_IMG.glob("*.jpg")):
    lp = TRAIN_LBL/(p.stem + ".txt")
    if not lp.exists(): continue
    W, H = Image.open(p).size
    boxes, cls = [], []
    for line in open(lp):
        f = line.split()
        if len(f) != 5: continue
        c = int(f[0]); cx, cy, bw, bh = map(float, f[1:])
        boxes.append([(cx-bw/2)*W,(cy-bh/2)*H,(cx+bw/2)*W,(cy+bh/2)*H]); cls.append(c)
    if boxes:
        gt_cache[p.name] = (np.array(boxes), np.array(cls), W, H)
print("GT cached for", len(gt_cache), "images")

def calibrate(seed):
    rid = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{seed}"
    det = pd.read_csv(FIGDATA/f"train_predictions_{rid}.csv")
    imgs = sorted(det.image.unique())[:CONFIG["CAL_IMAGES"]]
    nd = {c: np.zeros(len(GRID)-1) for c in NAMES}
    nh = {c: np.zeros(len(GRID)-1) for c in NAMES}
    sub = det[det.image.isin(imgs)]
    for img, d in sub.groupby("image"):
        if img not in gt_cache: continue
        gtb, gtc, _, _ = gt_cache[img]
        M = iou_matrix(d[["x1","y1","x2","y2"]].values, gtb)
        confs = d.conf.values; classes = d.class_id.values
        for c in np.unique(classes):
            m = classes == c
            same = gtc == c
            best = M[m][:, same].max(axis=1) if same.any() else np.zeros(int(m.sum()))
            idx = np.digitize(confs[m], GRID) - 1
            for b in range(len(GRID)-1):
                s = idx == b
                if s.any():
                    nd[c][b] += s.sum(); nh[c][b] += (best[s] >= CONFIG["IOU_MATCH"]).sum()
    centers = (GRID[:-1] + GRID[1:]) / 2
    rows = []
    for c in NAMES:
        p = nh[c]/np.maximum(nd[c], 1); ok = nd[c] >= CONFIG["MIN_BIN"]
        lo = hi = np.nan
        if ok.any():
            pv, cv = p[ok], centers[ok]
            lo = float(np.interp(CONFIG["PREC_LOW"],  pv, cv)) if pv.max() >= CONFIG["PREC_LOW"]  else float(cv[-1])
            hi = float(np.interp(CONFIG["PREC_HIGH"], pv, cv)) if pv.max() >= CONFIG["PREC_HIGH"] else float(cv[-1])
        rows.append({"seed": seed, "class_id": c, "class_name": NAMES[c],
                     "tau_low": lo, "tau_high": hi, "n_det": int(nd[c].sum())})
    return pd.DataFrame(rows)

bands = pd.concat([calibrate(s) for s in CONFIG["SEEDS"]], ignore_index=True)
bands.to_csv(FIGDATA/"tau_band_per_seed.csv", index=False)

agg = (bands.groupby(["class_id","class_name"])[["tau_low","tau_high"]]
       .agg(["mean","std"]).round(4))
print(agg.to_string())
print(f"\nmax seed-to-seed std: tau_low={bands.groupby('class_id').tau_low.std().max():.4f}"
      f"  tau_high={bands.groupby('class_id').tau_high.std().max():.4f}")


## [Cell 7] Crop writer
Context margin α expands each box by 20% on every side before the square crop is cut,
so the retrieved neighbour carries some surroundings rather than a bare object.

In [ ]:
# ===== Cell 7 =====
def write_crops(df, src_dir, out_dir, tag, size=None, quality=None):
    """df needs: image, x1, y1, x2, y2, class_id, conf. Returns the manifest."""
    size = size or CONFIG["CROP_SIZE"]; quality = quality or CONFIG["JPEG_Q"]
    out_dir.mkdir(parents=True, exist_ok=True)
    a = CONFIG["ALPHA"]
    rows, n_skip = [], 0
    for img, d in df.groupby("image", sort=False):
        p = Path(src_dir)/img
        try:
            im = Image.open(p).convert("RGB")
        except Exception:
            n_skip += len(d); continue
        W, H = im.size
        for i, r in enumerate(d.itertuples()):
            bw, bh = r.x2-r.x1, r.y2-r.y1
            if bw < CONFIG["MIN_CROP_PX"] or bh < CONFIG["MIN_CROP_PX"]:
                n_skip += 1; continue
            x1 = max(0, r.x1 - a*bw); y1 = max(0, r.y1 - a*bh)
            x2 = min(W, r.x2 + a*bw); y2 = min(H, r.y2 + a*bh)
            fn = f"{Path(img).stem}_{i:04d}.jpg"
            im.crop((x1, y1, x2, y2)).resize((size, size), Image.BILINEAR) \
              .save(out_dir/fn, quality=quality)
            rows.append({"crop": fn, "source_image": img, "class_id": int(r.class_id),
                         "class_name": NAMES.get(int(r.class_id), str(r.class_id)),
                         "conf": float(r.conf),
                         "x1": r.x1, "y1": r.y1, "x2": r.x2, "y2": r.y2,
                         "box_w": bw, "box_h": bh, "box_area": bw*bh, "set": tag})
    man = pd.DataFrame(rows)
    man.to_csv(out_dir.parent/f"manifest_{tag}.csv", index=False)
    return man, n_skip


## [Cell 9] Query crops — one set per seed
Each seed mines with its own band. Resumable: a seed whose manifest exists is skipped.

In [ ]:
# ===== Cell 9 =====
import time

band_mean = (bands.groupby("class_id")[["tau_low","tau_high"]].mean())
summary = []

for seed in CONFIG["SEEDS"]:
    rid = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{seed}"
    man_path = CROPS/f"manifest_queries_{rid}.csv"
    if man_path.exists():
        m = pd.read_csv(man_path)
        print(f"  {rid}: cached ({len(m):,} crops)")
        summary.append({"seed": seed, "crops": len(m)}); continue

    det = pd.read_csv(FIGDATA/f"train_predictions_{rid}.csv")
    b = bands[bands.seed == seed].set_index("class_id")
    det["tau_low"]  = det.class_id.map(b.tau_low).fillna(band_mean.tau_low.mean())
    det["tau_high"] = det.class_id.map(b.tau_high).fillna(band_mean.tau_high.mean())
    q = det[(det.conf >= det.tau_low) & (det.conf < det.tau_high)]

    t0 = time.time()
    man, skip = write_crops(q, TRAIN_IMG, CROPS/f"queries_{rid}", f"queries_{rid}")
    print(f"  {rid}: {len(man):,} crops ({skip:,} skipped) in {(time.time()-t0)/60:.1f} min")
    summary.append({"seed": seed, "crops": len(man)})

qs = pd.DataFrame(summary)
qs.to_csv(FIGDATA/"query_crop_counts.csv", index=False)
print(f"\ntotal query crops: {qs.crops.sum():,}")
print(qs.to_string(index=False))


## [Cell 11] Gallery — VisDrone test-dev, images only
Downloaded here for the first time. Its annotation folder is deleted immediately after
extraction so that no later cell can read it even by accident: the gallery must stay
label-free for the K3 claim to hold.

In [ ]:
# ===== Cell 11 =====
import shutil
from ultralytics.utils.downloads import download

GAL_ROOT = ROOT/"dataset"/"visdrone_gallery"
GAL_IMG = GAL_ROOT/"VisDrone2019-DET-test-dev"/"images"

if not GAL_IMG.exists():
    GAL_ROOT.mkdir(parents=True, exist_ok=True)
    download([CONFIG["GALLERY_URL"]], dir=GAL_ROOT, unzip=True, delete=False)

# enforce label-free gallery
for ann in GAL_ROOT.rglob("annotations"):
    if ann.is_dir():
        shutil.rmtree(ann); print("removed gallery annotations:", ann)
assert not list(GAL_ROOT.rglob("annotations")), "gallery labels still present"
assert not list(GAL_ROOT.rglob("labels")), "gallery labels still present"

n_gal = len(list(GAL_IMG.glob("*.jpg")))
print(f"gallery images: {n_gal}")
print("label dirs remaining:", [str(p) for p in GAL_ROOT.rglob('*label*')])


## [Cell 13] Gallery crops
Boxes come from the seed-42 baseline used purely as a region proposer. Only plausible
detections (conf ≥ 0.25) are kept: the gallery should hold recognisable objects, since it
is the pool queries are matched against.

In [ ]:
# ===== Cell 13 =====
from ultralytics import YOLO
import gc, torch

gal_man_path = CROPS/"manifest_gallery_visdrone.csv"
if gal_man_path.exists():
    gal_man = pd.read_csv(gal_man_path)
    print(f"cached gallery manifest: {len(gal_man):,} crops")
else:
    w = RUNS/f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s42"/"weights"/"best.pt"
    model = YOLO(str(w))
    imgs = sorted(GAL_IMG.glob("*.jpg"))
    rows = []
    for i in range(0, len(imgs), 250):
        for r in model.predict(source=[str(p) for p in imgs[i:i+250]],
                               imgsz=CONFIG["IMGSZ"], conf=CONFIG["GALLERY_CONF"],
                               max_det=CONFIG["GALLERY_MAX_DET"],
                               device=CONFIG["DEVICE"], stream=True, verbose=False):
            b = r.boxes
            if b is None or len(b) == 0: continue
            xyxy = b.xyxy.cpu().numpy(); cls = b.cls.cpu().numpy(); cf = b.conf.cpu().numpy()
            for k in range(len(cf)):
                rows.append((Path(r.path).name, int(cls[k]), float(cf[k]),
                             float(xyxy[k][0]), float(xyxy[k][1]),
                             float(xyxy[k][2]), float(xyxy[k][3])))
        print(f"    {min(i+250, len(imgs))}/{len(imgs)}  boxes={len(rows):,}", end="\r")
        gc.collect(); torch.cuda.empty_cache()

    gal_det = pd.DataFrame(rows, columns=["image","class_id","conf","x1","y1","x2","y2"])
    gal_man, skip = write_crops(gal_det, GAL_IMG, CROPS/"gallery_visdrone", "gallery_visdrone")
    print(f"\ngallery crops: {len(gal_man):,} ({skip:,} skipped)")

print(gal_man.class_name.value_counts().to_string())


## [Cell 15] Summary and disk check

In [ ]:
# ===== Cell 15 =====
import subprocess

tot_q = qs.crops.sum()
info = pd.DataFrame([
    ["query crops (all seeds)", f"{tot_q:,}"],
    ["query crops per seed (mean)", f"{qs.crops.mean():,.0f}"],
    ["gallery crops", f"{len(gal_man):,}"],
    ["gallery images", n_gal],
    ["context margin alpha", CONFIG["ALPHA"]],
    ["crop size", CONFIG["CROP_SIZE"]],
    ["band definition", f"precision {CONFIG['PREC_LOW']} -> {CONFIG['PREC_HIGH']}"],
], columns=["field","value"])
info.to_csv(FIGDATA/"crop_set_summary.csv", index=False)
print(info.to_string(index=False))
print()
print(subprocess.run(["du","-sh",str(CROPS)], capture_output=True, text=True).stdout)
print(subprocess.run(["df","-h",str(Path.home())], capture_output=True, text=True).stdout)


**Next:** `08_clip_faiss.ipynb` — embed every crop with CLIP ViT-B/32, build the FAISS
index over the gallery, run k-NN retrieval for each query set, and measure retrieval
quality (class-consistency@k) plus the UMAP projection for figure F10.

Report back: the per-seed band table, query crop counts, gallery crop count and disk usage.